# 03 — Train

Trains the Siamese U-Net with AMP, cosine schedule and early stopping on val mIoU. After every epoch the checkpoints are pushed to Drive — **if Colab disconnects, just re-run this notebook and training resumes** from `last.ckpt` (at most one epoch is lost).

**Storage contract** — code lives on GitHub, data & checkpoints live on Drive (`MyDrive/s2cd`), `/content` is disposable scratch. Every notebook is self-contained: run all cells top-to-bottom in a fresh GPU session.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os

REPO_URL = "https://github.com/<YOUR-GITHUB-USERNAME>/s2-change-detection.git"  # <-- EDIT once
REPO = "/content/s2-change-detection"
if os.path.exists(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone {REPO_URL} {REPO}
%cd {REPO}
!pip install -q -e .


In [ ]:
import torch
from s2cd.data import DriveStore
from s2cd.utils import get_device, describe_device, load_config

device = get_device()
print(describe_device(device))
store = DriveStore()  # Drive root: MyDrive/s2cd · scratch: /content


In [ ]:
from s2cd.train import train

train("configs/train.yaml", repo_root=".", store=store)


In [ ]:
import json
import matplotlib.pyplot as plt

run_dir = store.fetch_run("exp1")
history = json.loads((run_dir / "history.json").read_text())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot([h["train_loss"] for h in history], label="train")
ax1.plot([h["val_loss"] for h in history], label="val")
ax1.set_xlabel("epoch"); ax1.set_ylabel("loss"); ax1.legend()
ax2.plot([h["val_miou"] for h in history], label="val mIoU")
ax2.plot([h["val_change_iou"] for h in history], label="val change IoU")
ax2.set_xlabel("epoch"); ax2.legend()
plt.show()
